# Condition number with SymPy

In [ ]:
#    APM41012EP course notebook - Chapter 4 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Evaluation of the condition number of matrices with rational entries using SymPy
#    and computation of inverses - beware of the condition number evaluation with SciPy!
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import scipy.linalg
import sympy as sp
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "seaborn"

## Hilbert matrix

### Condition number of the Hilbert matrix

A Hilbert matrix is a square matrix with general term:

$$ H_{ij} = \frac{1}{i+j-1} $$

The Hilbert matrix of size 6 is written:


$$\begin{pmatrix}
 1 & \displaystyle \frac{1}{2} & \displaystyle \displaystyle \frac{1}{3} &\displaystyle  \frac{1}{4} & \displaystyle \frac{1}{5} & \displaystyle  \frac{1}{6} \\
\displaystyle \frac{1}{2} & \displaystyle \frac{1}{3} & \displaystyle \frac{1}{4} &\displaystyle  \frac{1}{5} & \displaystyle \frac{1}{6} & \displaystyle  \frac{1}{7} \\
\displaystyle \frac{1}{3} & \displaystyle \frac{1}{4} & \displaystyle \frac{1}{5} &\displaystyle  \frac{1}{6} & \displaystyle \frac{1}{7} & \displaystyle  \frac{1}{8} \\
\displaystyle \frac{1}{4} & \displaystyle \frac{1}{5} &\displaystyle \frac{1}{6} &\displaystyle  \frac{1}{7} & \displaystyle \frac{1}{8} & \displaystyle  \frac{1}{9} \\
\displaystyle \frac{1}{5} & \displaystyle \frac{1}{6} & \displaystyle \frac{1}{7} &\displaystyle  \frac{1}{8} & \displaystyle \frac{1}{9} & \displaystyle  \frac{1}{10} \\
\displaystyle \frac{1}{6} & \displaystyle \frac{1}{7} & \displaystyle \frac{1}{8} &\displaystyle  \frac{1}{9} & \displaystyle \frac{1}{10} & \displaystyle  \frac{1}{11} \\
\end{pmatrix}$$

In [ ]:
def show_cond_hilbert(n, show_coef=False):
    A = sp.Matrix(n, n, lambda i, j: sp.Rational(1, i + j + 1))
    print("-----------------------------------------------------------------------")
    print(f"Hilbert matrix of size {n}:")
    if (show_coef): sp.pprint(A, wrap_line=False)
    if (n < 8 and show_coef):
        print(f"Inverse of the Hilbert matrix of size {n}:")
        sp.pprint(A.inv(), wrap_line=False)
    c_inf = A.norm(sp.oo) * A.inv().norm(sp.oo)
    print(f"Condition number in the infinity norm        : {c_inf}")
    c_2 = np.linalg.norm(np.array(A.tolist(), dtype=np.float64), 2) * np.linalg.norm(np.array(A.inv().tolist(), dtype=np.float64), 2)
    print(f"Condition number in the 2-norm               : {c_2}")
    c_linalg = np.linalg.cond(scipy.linalg.hilbert(n))
    print(f"Condition number in the 2-norm (linalg.cond) : {c_linalg}\n")

for i in range(2, 15):
    show_cond_hilbert(i, True)

The reader can observe that:
- the condition number grows inordinately fast with the size $n$
- the condition number in the infinity norm is an **exact rational**: it is the product of
  two norms of matrices with rational coefficients, computed without any rounding
- the condition number computed with **linalg** is correct up to $n=12$ and wrong
  beyond (it relies on an evaluation of the eigenvalues that becomes grossly wrong
  because of the condition number of the matrix - see below)
- of course, the condition number depends on the norm being used

### Condition number and eigenvalues

The standard double-precision numerical tools cannot evaluate correctly the eigenvalues
of the matrix, and consequently the condition number in the sense of the 2-norm, beyond
n=13!

In [ ]:
n = np.arange(1, 21)

fig = go.Figure()

for i, ni in enumerate(n):
    eig_val = np.linalg.eigvals(scipy.linalg.hilbert(ni))
    eig_val = np.sort(eig_val)[::-1]
    A = sp.Matrix(ni, ni, lambda i, j: sp.Rational(1, i + j + 1))
    eig_val_sympy = sorted(A.charpoly().nroots(n=2*ni+30, maxsteps=100*ni), reverse=True)
    eig_val_sympy = np.array(eig_val_sympy, dtype=np.float64)
    #print(f"Eigenvalues of Hilbert matrix of size {ni} (SciPy): {eig_val}")
    #print(f"Eigenvalues of Hilbert matrix of size {ni} (SymPy): {eig_val_sympy}")
    fig.add_trace(go.Bar(visible=False, x=1.+np.arange(eig_val.size), y=np.abs(eig_val.real), name="SciPy"))
    fig.add_trace(go.Bar(visible=False, x=1.+np.arange(eig_val.size), y=np.abs(eig_val_sympy), name="SymPy"))

# display for n = 1
fig.data[0].visible = True
fig.data[1].visible = True

steps = []
for i, ni in enumerate(n):
    step = dict(method="update", label = f"{ni}", args=[{"visible": [(el==2*i) or (el==2*i+1) for el in range(len(fig.data))]}])
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'Matrix size = '}, steps=steps)]

legend = dict(orientation="h", y=1.1)
fig.update_layout(sliders=sliders, title = 'Representation of the eigenvalues', legend=legend)
fig.update_xaxes(range=[0,21], title="Eigenvalue index")
fig.update_yaxes(type="log", range=[-29, 1], exponentformat = 'e', title="Log of the eigenvalue")
fig.show()

Beyond the fifteenth eigenvalue, the evaluation by NumPy is wrong and impacts the
condition number, as can clearly be seen by comparing with the evaluation carried out from
the exact coefficients of SymPy, which captures the spectrum correctly.

### Perturbation of the right-hand side and solve (exactly!)

In a second step, we keep the same initial problem but we perturb the right-hand side
only, in the same spirit, that is with a rational perturbation of amplitude $1/1000000$.

In [ ]:
def diff_hilbert_b(n):
    print("-----------------------------------------------------------------------")
    print(f"Hilbert matrix of size {n}:")
    x = sp.Matrix([1 for i in range(n)])
    A = sp.Matrix(n, n, lambda i, j: sp.Rational(1, i + j + 1))
    print("Condition number in the infinity norm:", A.norm(sp.oo) * A.inv().norm(sp.oo))

    y = A * x
    print("We perturb the last component of the right-hand side by: ", sp.Rational(1, 10**6))

    y[n-1] *= (1 + sp.Rational(1, 1000000))   # perturbation of the right-hand side
    s = A.solve(y)                            # EXACT solve
    if (n <= 5):
        print("exact solution of the initial system   :", list(x))
        print("exact solution of the perturbed system :", list(s))
    err = max(abs(float(s[i] - x[i])) for i in range(n))
    print("Infinity norm of the difference between the two solutions: ", err, "\n")
    return err

for i in range(1, 15):
    diff_hilbert_b(i)

In this setting, we observe that as soon as $n=6$ the perturbed solution is very different
from the initial solution. Let us recall that we perform here an exact computation, which
makes it possible to evaluate the mathematical condition number of the problem of solving a
linear system whose matrix is a Hilbert matrix.

## Vandermonde matrix

The goal here is to observe the condition number of the Vandermonde matrix and its impact
on the solution of the corresponding linear system for a particular choice of right-hand
side.

The errors made because of the representation of real numbers on a machine in single and
double precision are evaluated using an exact solve with SymPy. We indeed show that the
errors are directly related to the condition number of the matrix when a solve with SciPy
is used.

In [ ]:
def vandermonde(n):
    X = np.array([i for i in range(1, n+1)])
    A = np.array([X**i for i in range(n)]).T
    return A

np.set_printoptions(linewidth=120)
print(vandermonde(10))

### Condition number and impact on the solve

In [ ]:
import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=LinAlgWarning)

def vandermonde_exact(x):   # here a list is expected
    n = len(x)
    return sp.Matrix(n, n, lambda i, j: sp.Integer(x[i])**j)

x_values = [1,2,3,4,5,6,7,8,9,10,11,12,13,14]

for i in range(4, 13):
    print("-------------------------------------------------")
    print("Vandermonde matrix of size: ", i)
    A = vandermonde_exact(x_values[1:i+1])
    cond = np.linalg.norm(np.array(A.tolist(), dtype=np.float64), 2) * np.linalg.norm(np.array(A.inv().tolist(), dtype=np.float64), 2)
    print("Condition number in the 2-norm:", cond)

    # exact RATIONAL right-hand side: [1/2, 1/2, 20, 1/2, ..., 1/2, -1/10]
    bn = [sp.Rational(1, 2)] * i
    bn[2] = sp.Integer(20)
    bn[-1] = sp.Rational(-1, 10)
    b = sp.Matrix(bn)

    x = A.solve(b)          # EXACT (rational) solution

    print("Solution of Ax = b:")
    bb64 = np.array(b.tolist(), dtype=np.float64).ravel()
    bb32 = bb64.astype(np.float32)
    AA64 = np.array(A.tolist(), dtype=np.float64)
    AA32 = AA64.astype(np.float32)

    x64 = scipy.linalg.solve(AA64, bb64)
    x32 = scipy.linalg.solve(AA32, bb32)

    xe = np.array(x.tolist(), dtype=np.float64).ravel()
    e64 = scipy.linalg.norm(xe - x64, np.inf)
    e32 = scipy.linalg.norm(xe - x32, np.inf)
    print("Infinity norm of the error (64-bit precision):", e64)
    print("Infinity norm of the error (32-bit precision):", e32)
    print("||error|| / cond (64-bit precision): ", e64/float(cond))
    print("||error|| / cond (32-bit precision): ", e32/float(cond))

## Laplacian matrix

The example of the Hilbert matrix is an extreme case that highlights well the difficulties
associated with an ill-conditioned matrix; that of the Vandermonde matrix, after the
chapters on interpolation and quadrature, is already more realistic and is encountered in
practice, even though there are ways of evaluating the weights exactly and thus of getting
rid of the errors associated with the poor conditioning and with the solution of the linear
system.

A classical case seen in the course on linear systems, and which we will have to solve, is
that of the solution of PDEs by centred finite differences and of the Laplacian matrix. In
this practical and classical case, we come back to the condition number of this type of
symmetric positive definite matrices and to its evaluation by the standard SciPy tools. It
clearly appears that in this case the condition number grows quadratically with the number
of points and not exponentially! Considering matrices of size several hundreds therefore
remains perfectly reasonable with the usual double-precision tools.

### Condition number as a function of the size

In [ ]:
def laplacian(n):
    M = -2*np.identity(n)
    for i in range(1, n):
        M[i,i-1] = 1
        M[i-1,i] = 1
    return M

def laplacian_exact(n):
    return sp.Matrix(n, n, lambda i, j: -2 if i == j else (1 if abs(i-j) == 1 else 0))

cond = np.zeros(40)
cond2 = np.zeros(40)
condinf = np.zeros(40)

for i in range(5, 205, 5):
    print("------------------------------------------------------")
    print("Laplacian matrix of size: ", i)

    MM = laplacian(i)
    AA = laplacian_exact(i)

    condinf[(i-4)//5] = AA.norm(sp.oo) * AA.inv().norm(sp.oo)
    cond2[(i-4)//5]   = np.linalg.norm(np.array(AA.tolist(), dtype=np.float64), 2) * np.linalg.norm(np.array(AA.inv().tolist(), dtype=np.float64), 2)

    print("Condition number in the infinity norm (exact) : ", condinf[(i-4)//5])
    print("Condition number in the 2-norm (sympy)        : ", cond2[(i-4)//5])
    print("Condition number in the 2-norm (linalg)       : ", np.linalg.cond(MM))
    cond[(i-4)//5] = (np.sin(i*np.pi/2/(i+1))/np.sin(np.pi/2/(i+1)))**2
    print("Exact condition number                        : ", cond[(i-4)//5])

### Estimation of the condition number

In [ ]:
nn = np.arange(5, 205, 5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=nn, y=cond, name='Exact condition number'))
fig.add_trace(go.Scatter(x=nn, y=4*(nn+1)**2/np.pi**2, name='(2(n+1)/pi)^2', line_dash='dash'))
fig.add_trace(go.Scatter(x=nn, y=condinf, name='Infinity-norm condition number'))
fig.update_layout(title="Condition number of the discretisation matrix of the Laplacian",
                  xaxis_title="Matrix size",
                  yaxis_title="Condition number")
fig.show()

For the Laplacian matrix with Dirichlet conditions on the interval $[0,1]$, a very precise
estimate of the condition number in the $2$-norm can be obtained: $4(n+1)^2/\pi^2$, as the
graph above shows.